[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notebooks/learning-guides/06-arboles-decision-regresion-entropia.ipynb)

# Árboles de decisión de regresión y entropía

**Objetivo del notebook**

En este notebook estudiaremos:

1. Qué significa la **entropía** desde la teoría de la información.
2. Cómo interpretar la entropía como la cantidad promedio de información necesaria para describir un sistema.
3. La relación entre entropía, incertidumbre y árboles de decisión.
4. La diferencia entre un árbol de **clasificación** y un árbol de **regresión**.
5. Cómo un árbol de regresión selecciona divisiones usando la reducción del error.
6. Cómo entrenar, visualizar e interpretar un `DecisionTreeRegressor` con Python.

> **Idea fundamental:** la entropía se utiliza principalmente como criterio en árboles de **clasificación**.  
> En los árboles de **regresión**, la variable objetivo es continua y normalmente se utilizan criterios como el **error cuadrático medio (MSE)** o la reducción de la varianza.



## 1. ¿Qué es la entropía?

En teoría de la información, la **entropía de Shannon** mide la incertidumbre promedio de un sistema o, de manera equivalente, la cantidad promedio de información necesaria para describir su resultado.

Sea una variable aleatoria \(X\) con posibles resultados \(x_1,\dots,x_n\), cuyas probabilidades son

\[
p_1,p_2,\dots,p_n.
\]

La entropía se define como:

\[
H(X)=-\sum_{i=1}^{n}p_i\log_2(p_i)
\]

y se mide en **bits** cuando usamos logaritmo en base 2.

### Interpretación

- **Entropía baja:** el sistema es predecible y necesitamos poca información para describir qué ocurrió.
- **Entropía alta:** existe mayor incertidumbre y necesitamos más información para identificar el resultado.

Por ejemplo:

- Si una moneda siempre cae en cara, no existe incertidumbre: \(H=0\).
- Si una moneda justa tiene 50 % de probabilidad de cara y 50 % de sello, la incertidumbre es máxima: \(H=1\) bit.


In [ ]:

import numpy as np

def entropia(probabilidades):
    # Calcula la entropía de Shannon en bits.
    p = np.asarray(probabilidades, dtype=float)
    p = p[p > 0]  # evita log2(0)
    return -np.sum(p * np.log2(p))

casos = {
    "Sistema completamente predecible": [1.0, 0.0],
    "Moneda sesgada": [0.9, 0.1],
    "Moneda justa": [0.5, 0.5],
    "Cuatro resultados equiprobables": [0.25, 0.25, 0.25, 0.25]
}

for nombre, probabilidades in casos.items():
    print(f"{nombre:35s} -> H = {entropia(probabilidades):.4f} bits")



### ¿Por qué una moneda justa tiene más entropía?

Antes de observar el lanzamiento de una moneda justa no sabemos cuál de los dos resultados ocurrirá. Ambos son igualmente plausibles.

En cambio, si la moneda produce cara el 100 % de las veces, conocer el resultado no aporta información nueva: ya sabíamos qué iba a ocurrir.

Por eso podemos pensar la entropía como una medida de:

> **cuánta información adicional necesitamos para describir el estado del sistema cuando todavía existe incertidumbre.**


In [ ]:

import matplotlib.pyplot as plt

p = np.linspace(0.001, 0.999, 500)
H = -(p * np.log2(p) + (1-p) * np.log2(1-p))

plt.figure(figsize=(8, 5))
plt.plot(p, H)
plt.axvline(0.5, linestyle="--", alpha=0.7)
plt.xlabel("Probabilidad de uno de los dos resultados, p")
plt.ylabel("Entropía H(p) [bits]")
plt.title("Entropía de un sistema binario")
plt.grid(alpha=0.25)
plt.show()



## 2. Entropía y árboles de decisión

Un árbol de decisión busca realizar preguntas que separen los datos en grupos cada vez más homogéneos.

### En clasificación

Si queremos predecir una **clase** —por ejemplo, `"Sí"` o `"No"`— podemos medir la impureza de un nodo con entropía.

Un nodo con una sola clase tiene entropía cercana a 0.

Un nodo con varias clases muy mezcladas tiene mayor entropía.

El árbol intenta encontrar una división que produzca la mayor **ganancia de información**:

\[
IG = H(\text{padre}) -
\left[
\frac{n_L}{n}H(L)+
\frac{n_R}{n}H(R)
\right]
\]

Es decir, queremos disminuir la incertidumbre después de realizar una pregunta.



## 3. ¿Qué ocurre en un árbol de regresión?

En **regresión** no queremos predecir una clase, sino un número continuo, por ejemplo:

- precio de una vivienda,
- temperatura,
- salario,
- concentración de una sustancia,
- supervivencia estimada,
- demanda de un producto.

Por esta razón, la entropía de clases **no es el criterio natural**.

En su lugar, un árbol de regresión busca divisiones que agrupen observaciones con valores de \(y\) similares.

Uno de los criterios más comunes es el **error cuadrático medio**:

\[
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\bar y)^2
\]

En cada nodo, la predicción del árbol es normalmente la **media** de los valores de \(y\) que llegaron a ese nodo.

El mejor corte es aquel que produce la mayor reducción del error.



## 4. Ejemplo intuitivo: precio de una vivienda según su tamaño

Crearemos datos simulados donde:

- \(X\): tamaño de la vivienda en metros cuadrados.
- \(y\): precio de la vivienda en millones de pesos.

La relación tendrá cierta estructura no lineal para que podamos observar cómo el árbol divide el espacio en regiones.


In [ ]:

import pandas as pd

rng = np.random.default_rng(42)

n = 120
tamano = np.sort(rng.uniform(40, 220, n))

precio = (
    80
    + 0.9 * tamano
    + 35 * (tamano > 100)
    + 45 * (tamano > 160)
    + rng.normal(0, 12, n)
)

df = pd.DataFrame({
    "tamano_m2": tamano,
    "precio_millones": precio
})

df.head()


In [ ]:

plt.figure(figsize=(9, 5))
plt.scatter(df["tamano_m2"], df["precio_millones"], alpha=0.75)
plt.xlabel("Tamaño de la vivienda (m²)")
plt.ylabel("Precio (millones de COP)")
plt.title("Datos simulados: tamaño vs. precio")
plt.grid(alpha=0.25)
plt.show()



## 5. ¿Cómo decide el árbol dónde cortar?

Supongamos que evaluamos una regla:

\[
\text{tamaño} \leq t
\]

Esa regla crea dos grupos:

- nodo izquierdo: viviendas con tamaño \(\leq t\),
- nodo derecho: viviendas con tamaño \(>t\).

Podemos calcular el error de ambos grupos y obtener el **error ponderado después del corte**:

\[
MSE_{\text{split}}
=
\frac{n_L}{n}MSE_L+
\frac{n_R}{n}MSE_R
\]

El árbol selecciona el corte que minimiza este valor, o equivalentemente, el que produce la mayor reducción frente al error del nodo padre.


In [ ]:

def mse(y):
    y = np.asarray(y)
    return np.mean((y - y.mean()) ** 2)

def evaluar_corte(x, y, umbral):
    x = np.asarray(x)
    y = np.asarray(y)

    izquierda = y[x <= umbral]
    derecha = y[x > umbral]

    if len(izquierda) == 0 or len(derecha) == 0:
        return np.inf

    mse_izq = mse(izquierda)
    mse_der = mse(derecha)

    mse_ponderado = (
        len(izquierda) / len(y) * mse_izq
        + len(derecha) / len(y) * mse_der
    )
    return mse_ponderado

X_manual = df["tamano_m2"].to_numpy()
y_manual = df["precio_millones"].to_numpy()

mse_padre = mse(y_manual)

umbrales = np.linspace(X_manual.min() + 1, X_manual.max() - 1, 200)
errores = np.array([evaluar_corte(X_manual, y_manual, u) for u in umbrales])

mejor_indice = np.argmin(errores)
mejor_umbral = umbrales[mejor_indice]
mejor_error = errores[mejor_indice]
reduccion_error = mse_padre - mejor_error

print(f"MSE del nodo padre:         {mse_padre:.2f}")
print(f"Mejor primer umbral aprox.: {mejor_umbral:.2f} m²")
print(f"MSE después del corte:      {mejor_error:.2f}")
print(f"Reducción del MSE:          {reduccion_error:.2f}")


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(umbrales, errores)
plt.axvline(mejor_umbral, linestyle="--", label=f"Mejor corte ≈ {mejor_umbral:.1f} m²")
plt.xlabel("Umbral candidato (m²)")
plt.ylabel("MSE ponderado después del corte")
plt.title("Búsqueda del mejor primer corte")
plt.legend()
plt.grid(alpha=0.25)
plt.show()



## 6. Entrenar un árbol de decisión de regresión

Ahora utilizaremos `DecisionTreeRegressor` de `scikit-learn`.

Usaremos una profundidad máxima pequeña para poder interpretar el árbol fácilmente.


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df[["tamano_m2"]]
y = df["precio_millones"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

modelo = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f} millones")
print(f"RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.2f} millones")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")



## 7. Visualización del árbol

En cada nodo veremos información como:

- `tamano_m2 <= ...`: regla utilizada para dividir.
- `squared_error`: variabilidad/error dentro del nodo.
- `samples`: número de observaciones.
- `value`: predicción del nodo, que corresponde aproximadamente a la media del precio en ese grupo.


In [ ]:

plt.figure(figsize=(18, 9))
plot_tree(
    modelo,
    feature_names=["tamano_m2"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árbol de decisión para regresión")
plt.show()



## 8. La predicción de un árbol es una función por escalones

Un árbol de regresión divide el eje de la variable explicativa en regiones.

Todas las observaciones que terminan en la misma hoja reciben la misma predicción. Por eso la función estimada tiene forma de **escalones**.


In [ ]:

x_grid = np.linspace(df["tamano_m2"].min(), df["tamano_m2"].max(), 500)
x_grid_df = pd.DataFrame({"tamano_m2": x_grid})
y_grid = modelo.predict(x_grid_df)

plt.figure(figsize=(10, 6))
plt.scatter(X_train["tamano_m2"], y_train, alpha=0.65, label="Entrenamiento")
plt.scatter(X_test["tamano_m2"], y_test, marker="x", s=70, label="Prueba")
plt.plot(x_grid, y_grid, linewidth=2.5, label="Predicción del árbol")
plt.xlabel("Tamaño de la vivienda (m²)")
plt.ylabel("Precio (millones de COP)")
plt.title("DecisionTreeRegressor: predicción por regiones")
plt.legend()
plt.grid(alpha=0.25)
plt.show()



## 9. Predicción de una nueva observación

Probemos una vivienda de **135 m²**.


In [ ]:

nueva_vivienda = pd.DataFrame({"tamano_m2": [135]})
prediccion = modelo.predict(nueva_vivienda)[0]

print("Para una vivienda de 135 m², el árbol predice:")
print(f"{prediccion:.2f} millones de COP")



## 10. Profundidad del árbol y sobreajuste

Si permitimos que el árbol crezca demasiado, puede generar muchas regiones pequeñas y memorizar el conjunto de entrenamiento.

Esto se conoce como **sobreajuste (overfitting)**.

Compararemos varios valores de `max_depth`.


In [ ]:

resultados = []

for profundidad in [1, 2, 3, 5, 8, None]:
    m = DecisionTreeRegressor(
        max_depth=profundidad,
        random_state=42
    )
    m.fit(X_train, y_train)

    pred_train = m.predict(X_train)
    pred_test = m.predict(X_test)

    resultados.append({
        "max_depth": str(profundidad),
        "R2_train": r2_score(y_train, pred_train),
        "R2_test": r2_score(y_test, pred_test),
        "RMSE_test": mean_squared_error(y_test, pred_test) ** 0.5
    })

pd.DataFrame(resultados)



## 11. Entropía vs. error cuadrático: idea clave

| Problema | Variable objetivo | Criterios comunes |
|---|---|---|
| Clasificación | Categórica | Entropía, ganancia de información, Gini |
| Regresión | Continua | Error cuadrático, error absoluto, Poisson |

### Conexión conceptual

Aunque el árbol de regresión no utilice normalmente entropía de Shannon, ambos enfoques comparten la misma filosofía:

> **buscar una pregunta que reduzca al máximo la incertidumbre o heterogeneidad del sistema.**

En clasificación, esa incertidumbre puede medirse mediante entropía.

En regresión, la heterogeneidad se cuantifica normalmente observando cuánto se dispersan los valores numéricos alrededor de su media.



## 12. Conclusiones

1. La **entropía** cuantifica la incertidumbre promedio o la cantidad promedio de información necesaria para describir un sistema.
2. Una distribución más impredecible tiene mayor entropía.
3. En árboles de clasificación, una división puede evaluarse por cuánto reduce la entropía.
4. En árboles de regresión, la variable objetivo es continua y normalmente se buscan divisiones que reduzcan el **MSE** o la varianza.
5. Cada hoja de un árbol de regresión genera una predicción constante, normalmente basada en la media de los valores de entrenamiento que pertenecen a esa hoja.
6. Árboles demasiado profundos pueden sobreajustar los datos.

### Preguntas para comprobar la comprensión

1. ¿Por qué la entropía de una moneda justa es mayor que la de una moneda que siempre cae en cara?
2. ¿Por qué la entropía no es el criterio estándar para un árbol de regresión?
3. ¿Qué representa el `value` de una hoja en `DecisionTreeRegressor`?
4. ¿Qué sucede cuando aumentamos demasiado la profundidad del árbol?
5. ¿Por qué la predicción de un árbol de regresión tiene forma de escalones?
